# CMI Kaggle Competition - Data Exploration
Test the preprocessing and feature engineering pipeline on sample data.

In [ ]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path

# Add project root to path
sys.path.insert(0, str(Path.cwd()))

from src.preprocessing import Preprocessor
from src.feature_engineering import FeatureEngineer
from config import IMU_FEATURES, PREPROCESSING_CONFIG

print('✅ All imports successful!')

## Load Sample Data
Load training data if available

In [ ]:
# Check if data exists
from config import DATA_RAW
train_file = DATA_RAW / 'train.csv'

if train_file.exists():
    df = pd.read_csv(train_file)
    print(f'Data loaded: {df.shape}')
    print(f'\nColumns: {list(df.columns)}')
    print(f'\nFirst rows:')
    print(df.head())
else:
    print(f'⚠️  Data file not found: {train_file}')
    print('Run: python3 -m src.pipeline --download')

## Test Preprocessing
Apply preprocessing steps to a single sequence

In [ ]:
if train_file.exists():
    # Get first sequence
    seq_id = df['sequence_id'].iloc[0]
    seq_data = df[df['sequence_id'] == seq_id].copy()
    
    print(f'Sequence: {seq_id}')
    print(f'Original shape: {seq_data.shape}')
    
    # Apply preprocessing
    preproc = Preprocessor()
    processed = seq_data.copy()
    processed = preproc.clean_tof(processed)
    processed = preproc.fill_values(processed)
    processed = preproc.pad_sequence(processed, PREPROCESSING_CONFIG['max_sequence_length'])
    
    print(f'After preprocessing: {processed.shape}')
    print(f'✅ Preprocessing successful')

## Test Feature Engineering
Create 35 IMU features from preprocessed data

In [ ]:
if train_file.exists():
    # Create features
    features = FeatureEngineer.create_imu_features(processed)
    
    print(f'Features created: {features.shape}')
    print(f'\nFeature list ({len(IMU_FEATURES)} total):')
    print(IMU_FEATURES)
    print(f'\nFirst feature row:')
    print(features.iloc[0])

## Statistics
Summary of feature engineering pipeline

In [ ]:
print('Pipeline Configuration:')
print(f'- Max sequence length: {PREPROCESSING_CONFIG["max_sequence_length"]}')
print(f'- Total IMU features: {len(IMU_FEATURES)}')
print(f'- Model: RandomForestClassifier(n_estimators=100, max_depth=10)')
print(f'\nIMU Features breakdown:')
print(f'  - Basic: acc_x, acc_y, acc_z, acc_mag, acc_mag_jerk (5)')
print(f'  - Jerk: jerk_x/y/z, jerk_magnitude (4)')
print(f'  - Correlations: acc_xy/xz/yz_corr (3)')
print(f'  - Rotation: rot_w/x/y/z, rot_angle, rot_angle_vel (6)')
print(f'  - Angular: angular_vel_x/y/z, magnitude, distance (5)')
print(f'  - Gravity-free variants: 12')
print(f'  Total: {5+4+3+6+5+12} features')